# 09 — Сводная статистика для дашборда: только готовые mart

Этот ноутбук **не читает `mart_events.parquet`** и не пересчитывает событийный слой.

Используются только готовые витрины:

- `mart_users.parquet`
- `mart_questions.parquet`
- `mart_topics.parquet`
- `mart_student_stage.parquet`

Это существенно снижает расход RAM: агрегации выполняются уже по компактным mart-таблицам.

## Важное ограничение

В текущих mart нет полей `session_start`, `session_end` или готовой
`session_duration`.

Поэтому **длительность отдельной сессии здесь не рассчитывается**:
восстанавливать её из агрегированных mart было бы некорректно.

Доступные близкие метрики:

- `sessions_count` — число сессий пользователя;
- `avg_questions_per_session` — среднее число вопросов за сессию;
- `learning_duration_days` — span активности пользователя;
- `median_time_between_events` — медианный интервал между событиями;
- `median_question_elapsed_time` — медианное время ответа на вопрос.

## Результаты

В `processed/dashboard_stats/` создаются:

1. `dashboard_kpi.parquet / .csv`
2. `dashboard_metric_summary.parquet / .csv`
3. `dashboard_stage_summary.parquet / .csv`
4. `dashboard_part_summary.parquet / .csv`
5. `dashboard_source_inventory.parquet / .csv`
6. `dashboard_metric_availability.parquet / .csv`

In [6]:
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


def find_data_dir() -> Path:
    candidates = [
        Path("data"),
        Path("../data"),
        Path("../../data"),
        Path("."),
        Path(".."),
    ]

    required = {
        "mart_users.parquet",
        "mart_questions.parquet",
        "mart_topics.parquet",
        "mart_student_stage.parquet",
    }

    for data_dir in candidates:
        processed = data_dir / "processed"
        if all((processed / name).exists() for name in required):
            return data_dir

    raise FileNotFoundError(
        "Не найдены все готовые mart в processed/. "
        "Нужны mart_users.parquet, mart_questions.parquet, "
        "mart_topics.parquet, mart_student_stage.parquet."
    )


DATA_DIR = find_data_dir()
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROCESSED_DIR / "dashboard_stats"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MART_USERS_PATH = PROCESSED_DIR / "mart_users.parquet"
MART_QUESTIONS_PATH = PROCESSED_DIR / "mart_questions.parquet"
MART_TOPICS_PATH = PROCESSED_DIR / "mart_topics.parquet"
MART_STAGE_PATH = PROCESSED_DIR / "mart_student_stage.parquet"

KPI_PATH = OUTPUT_DIR / "dashboard_kpi.parquet"
KPI_CSV_PATH = OUTPUT_DIR / "dashboard_kpi.csv"

METRIC_SUMMARY_PATH = OUTPUT_DIR / "dashboard_metric_summary.parquet"
METRIC_SUMMARY_CSV_PATH = OUTPUT_DIR / "dashboard_metric_summary.csv"

STAGE_SUMMARY_PATH = OUTPUT_DIR / "dashboard_stage_summary.parquet"
STAGE_SUMMARY_CSV_PATH = OUTPUT_DIR / "dashboard_stage_summary.csv"

PART_SUMMARY_PATH = OUTPUT_DIR / "dashboard_part_summary.parquet"
PART_SUMMARY_CSV_PATH = OUTPUT_DIR / "dashboard_part_summary.csv"

SOURCE_INVENTORY_PATH = OUTPUT_DIR / "dashboard_source_inventory.parquet"
SOURCE_INVENTORY_CSV_PATH = OUTPUT_DIR / "dashboard_source_inventory.csv"

AVAILABILITY_PATH = OUTPUT_DIR / "dashboard_metric_availability.parquet"
AVAILABILITY_CSV_PATH = OUTPUT_DIR / "dashboard_metric_availability.csv"


def sql_path(path: Path) -> str:
    return str(path.resolve()).replace("\\", "/").replace("'", "''")


con = duckdb.connect()

# Намеренно консервативные настройки:
# этот ноутбук работает только по компактным mart.
con.execute("SET threads = 2")
con.execute("SET memory_limit = '2GB'")
con.execute("SET preserve_insertion_order = false")


def parquet_columns(path: Path) -> list[str]:
    p = sql_path(path)
    return con.sql(
        f"DESCRIBE SELECT * FROM read_parquet('{p}')"
    ).df()["column_name"].tolist()


MART_USERS = sql_path(MART_USERS_PATH)
MART_QUESTIONS = sql_path(MART_QUESTIONS_PATH)
MART_TOPICS = sql_path(MART_TOPICS_PATH)
MART_STAGE = sql_path(MART_STAGE_PATH)

print("DATA_DIR:", DATA_DIR.resolve())
print("Output:", OUTPUT_DIR.resolve())
print("mart_events.parquet НЕ используется.")

DATA_DIR: /Users/anna/проект шар/data
Output: /Users/anna/проект шар/data/processed/dashboard_stats
mart_events.parquet НЕ используется.


## 1. Проверка схем готовых mart

In [7]:
expected = {
    "mart_users": {
        "user_id",
        "questions_count",
        "lectures_count",
        "events_count",
        "sessions_count",
        "unique_questions",
        "unique_parts",
        "unique_tags",
        "correct_answers",
        "incorrect_answers",
        "accuracy",
        "learning_duration_days",
        "avg_questions_per_session",
        "median_question_elapsed_time",
        "median_time_between_events",
        "lecture_views",
    },
    "mart_questions": {
        "question_id",
        "part",
        "attempts",
        "users_count",
        "correct_answers",
        "incorrect_answers",
        "accuracy",
        "difficulty",
        "median_elapsed_time",
        "mean_elapsed_time",
        "beginner_accuracy",
        "experienced_accuracy",
        "accuracy_gap",
        "enough_attempts",
        "fast_incorrect_rate",
        "slow_incorrect_rate",
        "error_streak_before_avg",
    },
    "mart_topics": {
        "tag",
        "questions_count",
        "attempts",
        "students_count",
        "correct_answers",
        "incorrect_answers",
        "accuracy",
        "difficulty",
        "median_elapsed_time",
        "lectures_count",
        "lecture_views",
        "accuracy_after_lecture",
        "accuracy_without_lecture",
        "accuracy_difference",
        "enough_attempts",
        "content_gap_flag",
    },
    "mart_student_stage": {
        "aggregation_level",
        "stage_order",
        "stage",
        "part",
        "students_count",
        "answers_count",
        "accuracy",
        "median_elapsed_time",
        "mean_elapsed_time",
        "lectures_count",
        "lecture_students_count",
        "lecture_rate",
        "sessions_count",
        "avg_questions_per_session",
        "small_sample_flag",
        "sample_size_status",
    },
}

paths = {
    "mart_users": MART_USERS_PATH,
    "mart_questions": MART_QUESTIONS_PATH,
    "mart_topics": MART_TOPICS_PATH,
    "mart_student_stage": MART_STAGE_PATH,
}

schema_rows = []

for source, path in paths.items():
    actual = set(parquet_columns(path))
    missing = sorted(expected[source] - actual)

    schema_rows.append({
        "source": source,
        "columns_count": len(actual),
        "missing_required_columns": ", ".join(missing),
        "schema_ok": len(missing) == 0,
    })

schema_check = pd.DataFrame(schema_rows)
display(schema_check)

bad_sources = schema_check.loc[
    ~schema_check["schema_ok"],
    ["source", "missing_required_columns"],
]

assert bad_sources.empty, (
    "В mart не хватает ожидаемых полей:\n"
    + bad_sources.to_string(index=False)
)

print("✅ Схемы всех четырёх mart подходят.")

,source,columns_count,missing_required_columns,schema_ok
0,mart_users,28,,True
1,mart_questions,20,,True
2,mart_topics,19,,True
3,mart_student_stage,24,,True


✅ Схемы всех четырёх mart подходят.


## 2. Инвентаризация готовых mart

In [8]:
inventory_rows = []

for source, path in paths.items():
    p = sql_path(path)

    rows_count = con.sql(
        f"SELECT COUNT(*) FROM read_parquet('{p}')"
    ).fetchone()[0]

    inventory_rows.append({
        "source": source,
        "file_name": path.name,
        "rows_count": int(rows_count),
        "columns_count": len(parquet_columns(path)),
        "size_mb": path.stat().st_size / 1024**2,
    })

source_inventory = pd.DataFrame(inventory_rows)

source_inventory.to_parquet(
    SOURCE_INVENTORY_PATH,
    index=False,
)
source_inventory.to_csv(
    SOURCE_INVENTORY_CSV_PATH,
    index=False,
)

display(source_inventory)

,source,file_name,rows_count,columns_count,size_mb
0,mart_users,mart_users.parquet,393656,28,22.551727
1,mart_questions,mart_questions.parquet,13523,20,1.007000
2,mart_topics,mart_topics.parquet,188,19,0.015323
3,mart_student_stage,mart_student_stage.parquet,56,24,0.020957


## 3. Основные KPI для карточек дашборда

In [9]:
def add_kpi(rows, section, metric, value, unit, description):
    rows.append({
        "section": section,
        "metric": metric,
        "value": None if pd.isna(value) else float(value),
        "unit": unit,
        "description": description,
    })


kpi_rows = []


# -------------------------
# Пользователи и активность
# -------------------------

u = con.sql(f"""
SELECT
    COUNT(*)::BIGINT AS users_count,

    SUM(questions_count)::BIGINT AS question_attempts,
    SUM(lectures_count)::BIGINT AS lecture_events,
    SUM(events_count)::BIGINT AS events_count,
    SUM(sessions_count)::BIGINT AS sessions_count,

    SUM(correct_answers)::BIGINT AS correct_answers,
    SUM(incorrect_answers)::BIGINT AS incorrect_answers,

    SUM(correct_answers)::DOUBLE
        / NULLIF(SUM(questions_count), 0)
        AS overall_accuracy,

    AVG(sessions_count)::DOUBLE
        AS mean_sessions_per_user,

    approx_quantile(sessions_count, 0.5)::DOUBLE
        AS median_sessions_per_user,

    AVG(unique_questions)::DOUBLE
        AS mean_unique_questions_per_user,

    approx_quantile(unique_questions, 0.5)::DOUBLE
        AS median_unique_questions_per_user,

    AVG(avg_questions_per_session)::DOUBLE
        AS mean_user_avg_questions_per_session,

    approx_quantile(avg_questions_per_session, 0.5)::DOUBLE
        AS median_user_avg_questions_per_session,

    AVG(learning_duration_days)::DOUBLE
        AS mean_learning_duration_days,

    approx_quantile(learning_duration_days, 0.5)::DOUBLE
        AS median_learning_duration_days,

    AVG(CAST(lecture_views > 0 AS INTEGER))::DOUBLE
        AS users_with_lecture_share,

    AVG(CAST(sessions_count = 1 AS INTEGER))::DOUBLE
        AS one_session_user_share

FROM read_parquet('{MART_USERS}')
""").df().iloc[0]


add_kpi(kpi_rows, "volume", "users_count",
        u["users_count"], "users",
        "Количество пользователей в mart_users.")

add_kpi(kpi_rows, "volume", "sessions_count",
        u["sessions_count"], "sessions",
        "Сумма sessions_count по пользователям.")

add_kpi(kpi_rows, "volume", "question_attempts",
        u["question_attempts"], "attempts",
        "Сумма вопросов/попыток по пользователям.")

add_kpi(kpi_rows, "volume", "lecture_events",
        u["lecture_events"], "events",
        "Сумма lecture events по пользователям.")

add_kpi(kpi_rows, "volume", "events_count",
        u["events_count"], "events",
        "Сумма всех событий по пользователям.")

add_kpi(kpi_rows, "quality", "overall_accuracy",
        u["overall_accuracy"], "share",
        "Взвешенная accuracy = correct_answers / questions_count.")

add_kpi(kpi_rows, "users", "mean_sessions_per_user",
        u["mean_sessions_per_user"], "sessions",
        "Среднее число сессий на пользователя.")

add_kpi(kpi_rows, "users", "median_sessions_per_user",
        u["median_sessions_per_user"], "sessions",
        "Медианное число сессий на пользователя.")

add_kpi(kpi_rows, "users", "mean_unique_questions_per_user",
        u["mean_unique_questions_per_user"], "questions",
        "Среднее число уникальных вопросов на пользователя.")

add_kpi(kpi_rows, "users", "median_unique_questions_per_user",
        u["median_unique_questions_per_user"], "questions",
        "Медианное число уникальных вопросов на пользователя.")

add_kpi(kpi_rows, "sessions", "mean_user_avg_questions_per_session",
        u["mean_user_avg_questions_per_session"], "questions",
        "Среднее пользовательских avg_questions_per_session.")

add_kpi(kpi_rows, "sessions", "median_user_avg_questions_per_session",
        u["median_user_avg_questions_per_session"], "questions",
        "Медиана пользовательских avg_questions_per_session.")

add_kpi(kpi_rows, "users", "mean_learning_duration_days",
        u["mean_learning_duration_days"], "days",
        "Средний span активности пользователя.")

add_kpi(kpi_rows, "users", "median_learning_duration_days",
        u["median_learning_duration_days"], "days",
        "Медианный span активности пользователя.")

add_kpi(kpi_rows, "users", "users_with_lecture_share",
        u["users_with_lecture_share"], "share",
        "Доля пользователей хотя бы с одной лекцией.")

add_kpi(kpi_rows, "users", "one_session_user_share",
        u["one_session_user_share"], "share",
        "Доля пользователей только с одной сессией.")


# -------------------------
# Вопросы
# -------------------------

q = con.sql(f"""
SELECT
    COUNT(*)::BIGINT AS questions_catalog,

    COUNT(*) FILTER (
        WHERE attempts > 0
    )::BIGINT AS questions_seen,

    COUNT(*) FILTER (
        WHERE enough_attempts
    )::BIGINT AS questions_enough_attempts,

    SUM(attempts)::BIGINT AS attempts_from_question_mart,

    SUM(correct_answers)::DOUBLE
        / NULLIF(SUM(attempts), 0)
        AS weighted_question_accuracy,

    AVG(accuracy) FILTER (
        WHERE accuracy IS NOT NULL
    )::DOUBLE AS mean_question_accuracy,

    AVG(difficulty) FILTER (
        WHERE difficulty IS NOT NULL
    )::DOUBLE AS mean_question_difficulty

FROM read_parquet('{MART_QUESTIONS}')
""").df().iloc[0]

add_kpi(kpi_rows, "questions", "questions_catalog",
        q["questions_catalog"], "questions",
        "Количество question_id в каталоге mart_questions.")

add_kpi(kpi_rows, "questions", "questions_seen",
        q["questions_seen"], "questions",
        "Количество вопросов хотя бы с одной попыткой.")

add_kpi(kpi_rows, "questions", "questions_enough_attempts",
        q["questions_enough_attempts"], "questions",
        "Количество вопросов с enough_attempts=True.")

add_kpi(kpi_rows, "questions", "weighted_question_accuracy",
        q["weighted_question_accuracy"], "share",
        "Взвешенная accuracy по mart_questions.")

add_kpi(kpi_rows, "questions", "mean_question_difficulty",
        q["mean_question_difficulty"], "share",
        "Средняя difficulty по вопросам.")


# -------------------------
# Темы
# -------------------------

t = con.sql(f"""
SELECT
    COUNT(*)::BIGINT AS topics_count,

    COUNT(*) FILTER (
        WHERE attempts > 0
    )::BIGINT AS topics_seen,

    COUNT(*) FILTER (
        WHERE enough_attempts
    )::BIGINT AS topics_enough_attempts,

    COUNT(*) FILTER (
        WHERE content_gap_flag
    )::BIGINT AS content_gap_topics,

    AVG(CAST(content_gap_flag AS INTEGER))::DOUBLE
        AS content_gap_share

FROM read_parquet('{MART_TOPICS}')
""").df().iloc[0]

add_kpi(kpi_rows, "topics", "topics_count",
        t["topics_count"], "topics",
        "Количество tag в mart_topics.")

add_kpi(kpi_rows, "topics", "topics_seen",
        t["topics_seen"], "topics",
        "Количество тем с попытками.")

add_kpi(kpi_rows, "topics", "topics_enough_attempts",
        t["topics_enough_attempts"], "topics",
        "Количество тем с enough_attempts=True.")

add_kpi(kpi_rows, "topics", "content_gap_topics",
        t["content_gap_topics"], "topics",
        "Количество тем с content_gap_flag=True.")

add_kpi(kpi_rows, "topics", "content_gap_share",
        t["content_gap_share"], "share",
        "Доля тем с content_gap_flag=True.")


# -------------------------
# Этапы обучения
# -------------------------

stage = con.sql(f"""
SELECT
    MAX(students_count) FILTER (
        WHERE aggregation_level = 'stage'
          AND stage_order = 1
    ) AS stage_1_users,

    MAX(students_count) FILTER (
        WHERE aggregation_level = 'stage'
          AND stage_order = 7
    ) AS stage_7_users

FROM read_parquet('{MART_STAGE}')
""").df().iloc[0]

if pd.notna(stage["stage_1_users"]) and stage["stage_1_users"] > 0:
    add_kpi(kpi_rows, "funnel", "users_reached_501plus",
            stage["stage_7_users"], "users",
            "Количество пользователей на этапе 501+.")

    add_kpi(kpi_rows, "funnel", "users_reached_501plus_share",
            stage["stage_7_users"] / stage["stage_1_users"], "share",
            "Доля пользователей первого этапа, дошедших до 501+.")


dashboard_kpi = pd.DataFrame(kpi_rows)

dashboard_kpi.to_parquet(KPI_PATH, index=False)
dashboard_kpi.to_csv(KPI_CSV_PATH, index=False)

display(dashboard_kpi)

,section,metric,value,unit,description
0,volume,users_count,3.936560e+05,users,Количество пользователей в mart_users.
1,volume,sessions_count,5.945418e+06,sessions,Сумма sessions_count по пользователям.
2,volume,question_attempts,9.927130e+07,attempts,Сумма вопросов/попыток по пользователям.
3,volume,lecture_events,1.959032e+06,events,Сумма lecture events по пользователям.
4,volume,events_count,1.012303e+08,events,Сумма всех событий по пользователям.
5,quality,overall_accuracy,6.572355e-01,share,Взвешенная accuracy = correct_answers / questi...
6,users,mean_sessions_per_user,1.510308e+01,sessions,Среднее число сессий на пользователя.
7,users,median_sessions_per_user,2.000000e+00,sessions,Медианное число сессий на пользователя.
8,users,mean_unique_questions_per_user,2.206674e+02,questions,Среднее число уникальных вопросов на пользоват...
9,users,median_unique_questions_per_user,4.000000e+01,questions,Медианное число уникальных вопросов на пользов...


## 4. Распределения метрик

Для числовых полей считаются:

`n`, `mean`, `stddev`, `min`, `p25`, `median`, `p75`, `p90`, `p95`, `max`.

Расчёт идёт отдельно по уровням `user`, `question`, `topic`.
`mart_student_stage` уже агрегирована и вынесена отдельно.

In [10]:
METRICS = {
    "user": {
        "path": MART_USERS_PATH,
        "metrics": {
            "questions_count": ("attempts", "Количество вопросных попыток пользователя."),
            "lectures_count": ("events", "Количество lecture events пользователя."),
            "events_count": ("events", "Количество всех событий пользователя."),
            "sessions_count": ("sessions", "Количество сессий пользователя."),
            "unique_questions": ("questions", "Уникальные вопросы пользователя."),
            "unique_parts": ("parts", "Уникальные part пользователя."),
            "unique_tags": ("topics", "Уникальные tags пользователя."),
            "accuracy": ("share", "Accuracy пользователя."),
            "progress_20": ("share_delta", "last_20_accuracy - first_20_accuracy."),
            "median_question_elapsed_time": ("ms", "Медианное время ответа пользователя."),
            "mean_question_elapsed_time": ("ms", "Среднее время ответа пользователя."),
            "median_time_between_events": ("ms", "Медианный интервал между событиями пользователя."),
            "learning_duration_days": ("days", "Span активности пользователя."),
            "avg_questions_per_session": ("questions", "Среднее число вопросов за сессию пользователя."),
            "lecture_views": ("events", "Lecture views пользователя."),
            "lecture_per_question": ("ratio", "Отношение лекций к вопросам."),
            "explanation_rate": ("share", "Доля событий с объяснением."),
        },
    },

    "question": {
        "path": MART_QUESTIONS_PATH,
        "metrics": {
            "attempts": ("attempts", "Количество попыток по вопросу."),
            "users_count": ("users", "Количество пользователей вопроса."),
            "accuracy": ("share", "Accuracy вопроса."),
            "difficulty": ("share", "1 - accuracy."),
            "median_elapsed_time": ("ms", "Медианное время ответа по вопросу."),
            "mean_elapsed_time": ("ms", "Среднее время ответа по вопросу."),
            "beginner_accuracy": ("share", "Accuracy начинающих."),
            "experienced_accuracy": ("share", "Accuracy опытных."),
            "accuracy_gap": ("share_delta", "experienced_accuracy - beginner_accuracy."),
            "fast_incorrect_rate": ("share", "Доля быстрых неправильных ответов."),
            "slow_incorrect_rate": ("share", "Доля медленных неправильных ответов."),
            "error_streak_before_avg": ("count", "Средний error streak перед вопросом."),
        },
    },

    "topic": {
        "path": MART_TOPICS_PATH,
        "metrics": {
            "questions_count": ("questions", "Число вопросов темы."),
            "attempts": ("attempts", "Число попыток темы."),
            "students_count": ("users", "Число пользователей темы."),
            "accuracy": ("share", "Accuracy темы."),
            "difficulty": ("share", "1 - accuracy темы."),
            "median_elapsed_time": ("ms", "Медианное время ответа по теме."),
            "lectures_count": ("lectures", "Число лекций в каталоге темы."),
            "lecture_views": ("events", "Число просмотров лекций темы."),
            "accuracy_after_lecture": ("share", "Accuracy после лекции по той же теме."),
            "accuracy_without_lecture": ("share", "Accuracy без предшествующей лекции по теме."),
            "accuracy_difference": ("share_delta", "Разница after lecture - without lecture."),
        },
    },
}


def summarize_metric(path: Path, entity_level: str, metric: str,
                     unit: str, description: str) -> pd.DataFrame:

    p = sql_path(path)

    stats = con.sql(f"""
    SELECT
        COUNT({metric})::BIGINT AS n,
        AVG({metric})::DOUBLE AS mean,
        stddev_samp({metric})::DOUBLE AS stddev,
        MIN({metric})::DOUBLE AS min,
        approx_quantile({metric}, 0.25)::DOUBLE AS p25,
        approx_quantile({metric}, 0.50)::DOUBLE AS median,
        approx_quantile({metric}, 0.75)::DOUBLE AS p75,
        approx_quantile({metric}, 0.90)::DOUBLE AS p90,
        approx_quantile({metric}, 0.95)::DOUBLE AS p95,
        MAX({metric})::DOUBLE AS max
    FROM read_parquet('{p}')
    WHERE {metric} IS NOT NULL
    """).df()

    stats.insert(0, "description", description)
    stats.insert(0, "unit", unit)
    stats.insert(0, "metric", metric)
    stats.insert(0, "entity_level", entity_level)

    return stats


summary_frames = []

for entity_level, config in METRICS.items():
    available = set(parquet_columns(config["path"]))

    for metric, (unit, description) in config["metrics"].items():
        if metric not in available:
            continue

        summary_frames.append(
            summarize_metric(
                config["path"],
                entity_level,
                metric,
                unit,
                description,
            )
        )

dashboard_metric_summary = pd.concat(
    summary_frames,
    ignore_index=True,
)

dashboard_metric_summary.to_parquet(
    METRIC_SUMMARY_PATH,
    index=False,
)
dashboard_metric_summary.to_csv(
    METRIC_SUMMARY_CSV_PATH,
    index=False,
)

display(dashboard_metric_summary)

,entity_level,metric,unit,description,n,mean,stddev,min,p25,median,p75,p90,p95,max
0,user,questions_count,attempts,Количество вопросных попыток пользователя.,393656,2.521778e+02,7.347211e+02,1.000000,30.000000,41.000000,1.540000e+02,5.790000e+02,1.150000e+03,1.760900e+04
1,user,lectures_count,events,Количество lecture events пользователя.,393656,4.976507e+00,1.596479e+01,0.000000,0.000000,0.000000,2.000000e+00,1.300000e+01,2.800000e+01,3.970000e+02
2,user,events_count,events,Количество всех событий пользователя.,393656,2.571543e+02,7.475509e+02,1.000000,30.000000,41.000000,1.570000e+02,5.920000e+02,1.180000e+03,1.791700e+04
3,user,sessions_count,sessions,Количество сессий пользователя.,393656,1.510308e+01,4.454433e+01,1.000000,1.000000,2.000000,1.000000e+01,3.600000e+01,6.900000e+01,2.374000e+03
4,user,unique_questions,questions,Уникальные вопросы пользователя.,393656,2.206674e+02,5.899140e+02,1.000000,29.000000,40.000000,1.420000e+02,5.150000e+02,1.019000e+03,1.121800e+04
5,user,unique_parts,parts,Уникальные part пользователя.,393656,4.639320e+00,2.309854e+00,1.000000,2.000000,5.000000,7.000000e+00,7.000000e+00,7.000000e+00,7.000000e+00
6,user,unique_tags,topics,Уникальные tags пользователя.,393656,6.173028e+01,4.357786e+01,1.000000,34.000000,43.000000,8.500000e+01,1.330000e+02,1.570000e+02,1.880000e+02
7,user,accuracy,share,Accuracy пользователя.,393656,5.451833e-01,1.633780e-01,0.000000,0.433580,0.570306,6.669302e-01,7.361547e-01,7.741746e-01,1.000000e+00
8,user,progress_20,share_delta,last_20_accuracy - first_20_accuracy.,209268,8.813483e-02,2.198800e-01,-0.900000,-0.050000,0.100000,2.498244e-01,3.500000e-01,4.499808e-01,9.500000e-01
9,user,median_question_elapsed_time,ms,Медианное время ответа пользователя.,393569,2.121571e+04,6.692793e+03,0.000000,17547.789983,20061.912720,2.399983e+04,2.864775e+04,3.246410e+04,3.000000e+05


## 5. Готовая таблица по этапам обучения

In [11]:
stage_df = con.sql(f"""
SELECT *
FROM read_parquet('{MART_STAGE}')
ORDER BY aggregation_level, stage_order, part
""").df()

first_stage_students = stage_df.loc[
    (stage_df["aggregation_level"] == "stage")
    & (stage_df["stage_order"] == 1),
    "students_count",
]

if not first_stage_students.empty:
    base_users = float(first_stage_students.iloc[0])

    stage_df["reach_from_stage_1_share"] = np.where(
        stage_df["aggregation_level"].eq("stage"),
        stage_df["students_count"] / base_users,
        np.nan,
    )
else:
    stage_df["reach_from_stage_1_share"] = np.nan

stage_df.to_parquet(STAGE_SUMMARY_PATH, index=False)
stage_df.to_csv(STAGE_SUMMARY_CSV_PATH, index=False)

display(
    stage_df[
        stage_df["aggregation_level"].eq("stage")
    ][[
        "stage_order",
        "stage",
        "students_count",
        "reach_from_stage_1_share",
        "answers_count",
        "accuracy",
        "median_elapsed_time",
        "lecture_rate",
        "sessions_count",
        "avg_questions_per_session",
    ]]
)

,stage_order,stage,students_count,reach_from_stage_1_share,answers_count,accuracy,median_elapsed_time,lecture_rate,sessions_count,avg_questions_per_session
0,1,1–10,393656,1.000000,3928411,0.507141,21232.034384,0.005162,475856,8.255462
1,2,11–20,390687,0.992458,3672430,0.502691,19999.546087,0.046400,482973,7.603800
2,3,21–50,332257,0.844029,7224103,0.585427,20028.406350,0.197817,682931,10.578086
3,4,51–100,174074,0.442198,7253473,0.644936,19999.712039,0.479509,633421,11.451267
4,5,101–200,123379,0.313418,10137455,0.654587,20488.588795,0.662212,780394,12.990175
5,6,201–500,84659,0.215058,18144113,0.664490,21008.151010,0.775039,1248021,14.538307
6,7,501+,44266,0.112448,48911315,0.691182,21175.349982,0.835427,2716513,18.005183


## 6. Сводка по `part` из mart_questions

In [12]:
part_summary = con.sql(f"""
SELECT
    part,

    COUNT(*)::BIGINT AS questions_catalog,

    COUNT(*) FILTER (
        WHERE attempts > 0
    )::BIGINT AS questions_seen,

    SUM(attempts)::BIGINT AS attempts,

    SUM(users_count)::BIGINT AS question_user_pairs,

    SUM(correct_answers)::BIGINT AS correct_answers,

    SUM(incorrect_answers)::BIGINT AS incorrect_answers,

    SUM(correct_answers)::DOUBLE
        / NULLIF(SUM(attempts), 0)
        AS weighted_accuracy,

    1.0 - (
        SUM(correct_answers)::DOUBLE
        / NULLIF(SUM(attempts), 0)
    ) AS weighted_difficulty,

    AVG(accuracy) FILTER (
        WHERE accuracy IS NOT NULL
    )::DOUBLE AS mean_question_accuracy,

    approx_quantile(
        median_elapsed_time,
        0.5
    ) FILTER (
        WHERE median_elapsed_time IS NOT NULL
    )::DOUBLE AS median_question_elapsed_time,

    approx_quantile(
        attempts,
        0.5
    )::DOUBLE AS median_attempts_per_question,

    COUNT(*) FILTER (
        WHERE enough_attempts
    )::BIGINT AS questions_enough_attempts

FROM read_parquet('{MART_QUESTIONS}')
GROUP BY part
ORDER BY part
""").df()

part_summary.to_parquet(PART_SUMMARY_PATH, index=False)
part_summary.to_csv(PART_SUMMARY_CSV_PATH, index=False)

display(part_summary)

,part,questions_catalog,questions_seen,attempts,question_user_pairs,correct_answers,incorrect_answers,weighted_accuracy,weighted_difficulty,mean_question_accuracy,median_question_elapsed_time,median_attempts_per_question,questions_enough_attempts
0,1,992,992,7454570,6735835,5553896,1900674,0.745032,0.254968,0.815104,21000.000000,5948.0,642
1,2,1647,1647,18743404,15706116,13283339,5460065,0.708694,0.291306,0.745296,17000.000000,9793.0,1602
2,3,1562,1562,8639907,7706350,6060514,2579393,0.701456,0.298544,0.744598,24000.000000,4951.0,1238
3,4,1439,1439,8067676,7279636,5090684,2976992,0.630998,0.369002,0.724918,24000.000000,4517.0,1040
4,5,5511,5511,40908153,35675224,24957570,15950583,0.610088,0.389912,0.666119,20000.000000,4617.0,5372
5,6,1212,1212,10501472,9064117,7029563,3471909,0.669388,0.330612,0.699596,29860.294118,5179.0,1207
6,7,1160,1160,4956118,4699753,3269061,1687057,0.659601,0.340399,0.717958,54642.288889,2762.0,1070


## 7. Какие метрики доступны без mart_events

In [13]:
metric_availability = pd.DataFrame([
    {
        "metric": "users_count",
        "available": True,
        "source": "mart_users",
        "comment": "COUNT(*) по пользователям.",
    },
    {
        "metric": "sessions_count",
        "available": True,
        "source": "mart_users.sessions_count",
        "comment": "Суммарное число сессий можно получить без событий.",
    },
    {
        "metric": "avg_questions_per_session",
        "available": True,
        "source": "mart_users / mart_student_stage",
        "comment": "Есть на уровне пользователя и этапа.",
    },
    {
        "metric": "unique_questions_per_user",
        "available": True,
        "source": "mart_users.unique_questions",
        "comment": "Можно строить распределение по пользователям.",
    },
    {
        "metric": "unique_questions_catalog",
        "available": True,
        "source": "mart_questions.question_id",
        "comment": "Одна строка = один question_id.",
    },
    {
        "metric": "question_accuracy_difficulty",
        "available": True,
        "source": "mart_questions",
        "comment": "Есть готовые accuracy/difficulty и количество попыток.",
    },
    {
        "metric": "topic_statistics",
        "available": True,
        "source": "mart_topics",
        "comment": "Есть attempts, students, accuracy, lectures и content_gap.",
    },
    {
        "metric": "stage_funnel",
        "available": True,
        "source": "mart_student_stage",
        "comment": "Есть students_count по этапам.",
    },
    {
        "metric": "user_learning_duration",
        "available": True,
        "source": "mart_users.learning_duration_days",
        "comment": "Это span всей активности пользователя, не отдельной сессии.",
    },
    {
        "metric": "session_duration",
        "available": False,
        "source": "",
        "comment": (
            "В текущих mart нет start/end или duration отдельной сессии. "
            "Без mart_events восстановить корректно нельзя."
        ),
    },
    {
        "metric": "questions_per_individual_session_distribution",
        "available": False,
        "source": "",
        "comment": (
            "Есть только avg_questions_per_session пользователя/этапа, "
            "но нет одной строки на отдельную сессию."
        ),
    },
])

metric_availability.to_parquet(
    AVAILABILITY_PATH,
    index=False,
)
metric_availability.to_csv(
    AVAILABILITY_CSV_PATH,
    index=False,
)

display(metric_availability)

,metric,available,source,comment
0,users_count,True,mart_users,COUNT(*) по пользователям.
1,sessions_count,True,mart_users.sessions_count,Суммарное число сессий можно получить без собы...
2,avg_questions_per_session,True,mart_users / mart_student_stage,Есть на уровне пользователя и этапа.
3,unique_questions_per_user,True,mart_users.unique_questions,Можно строить распределение по пользователям.
4,unique_questions_catalog,True,mart_questions.question_id,Одна строка = один question_id.
5,question_accuracy_difficulty,True,mart_questions,Есть готовые accuracy/difficulty и количество ...
6,topic_statistics,True,mart_topics,"Есть attempts, students, accuracy, lectures и ..."
7,stage_funnel,True,mart_student_stage,Есть students_count по этапам.
8,user_learning_duration,True,mart_users.learning_duration_days,"Это span всей активности пользователя, не отде..."
9,session_duration,False,,В текущих mart нет start/end или duration отде...


## 8. DQ: сверка итоговых чисел между mart

In [14]:
dq = {}

dq["users_rows"] = con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{MART_USERS}')
""").fetchone()[0]

dq["users_unique"] = con.sql(f"""
SELECT COUNT(DISTINCT user_id)
FROM read_parquet('{MART_USERS}')
""").fetchone()[0]

dq["questions_rows"] = con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{MART_QUESTIONS}')
""").fetchone()[0]

dq["questions_unique"] = con.sql(f"""
SELECT COUNT(DISTINCT question_id)
FROM read_parquet('{MART_QUESTIONS}')
""").fetchone()[0]

dq["topics_rows"] = con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{MART_TOPICS}')
""").fetchone()[0]

dq["topics_unique"] = con.sql(f"""
SELECT COUNT(DISTINCT tag)
FROM read_parquet('{MART_TOPICS}')
""").fetchone()[0]

dq["attempts_users"] = con.sql(f"""
SELECT SUM(questions_count)
FROM read_parquet('{MART_USERS}')
""").fetchone()[0]

dq["attempts_questions"] = con.sql(f"""
SELECT SUM(attempts)
FROM read_parquet('{MART_QUESTIONS}')
""").fetchone()[0]

dq_df = pd.DataFrame(
    [{"check": key, "value": value} for key, value in dq.items()]
)

display(dq_df)

assert dq["users_rows"] == dq["users_unique"], (
    "mart_users: user_id не уникален."
)

assert dq["questions_rows"] == dq["questions_unique"], (
    "mart_questions: question_id не уникален."
)

assert dq["topics_rows"] == dq["topics_unique"], (
    "mart_topics: tag не уникален."
)

if dq["attempts_users"] != dq["attempts_questions"]:
    print(
        "⚠️ Число attempts отличается между mart_users и mart_questions:",
        f"{dq['attempts_users']:,} vs {dq['attempts_questions']:,}."
    )
    print(
        "Это не останавливает расчёт: возможна разница в правилах "
        "фильтрации при построении двух mart."
    )
else:
    print("✅ Число question attempts между mart_users и mart_questions совпадает.")

print("✅ Проверки уникальности ключей mart пройдены.")

,check,value
0,users_rows,393656
1,users_unique,393656
2,questions_rows,13523
3,questions_unique,13523
4,topics_rows,188
5,topics_unique,188
6,attempts_users,99271300
7,attempts_questions,99271300


✅ Число question attempts между mart_users и mart_questions совпадает.
✅ Проверки уникальности ключей mart пройдены.


## 9. Итоговые файлы

In [15]:
outputs = [
    KPI_PATH,
    KPI_CSV_PATH,
    METRIC_SUMMARY_PATH,
    METRIC_SUMMARY_CSV_PATH,
    STAGE_SUMMARY_PATH,
    STAGE_SUMMARY_CSV_PATH,
    PART_SUMMARY_PATH,
    PART_SUMMARY_CSV_PATH,
    SOURCE_INVENTORY_PATH,
    SOURCE_INVENTORY_CSV_PATH,
    AVAILABILITY_PATH,
    AVAILABILITY_CSV_PATH,
]

for path in outputs:
    assert path.exists(), f"Не создан: {path}"

    print(
        f"✅ {path.name:<44} "
        f"{path.stat().st_size / 1024**2:,.3f} MB"
    )

print()
print("✅ Сводная статистика построена только на готовых mart.")
print("✅ mart_events.parquet не читался.")

✅ dashboard_kpi.parquet                        0.005 MB
✅ dashboard_kpi.csv                            0.003 MB
✅ dashboard_metric_summary.parquet             0.012 MB
✅ dashboard_metric_summary.csv                 0.008 MB
✅ dashboard_stage_summary.parquet              0.021 MB
✅ dashboard_stage_summary.csv                  0.012 MB
✅ dashboard_part_summary.parquet               0.009 MB
✅ dashboard_part_summary.csv                   0.001 MB
✅ dashboard_source_inventory.parquet           0.004 MB
✅ dashboard_source_inventory.csv               0.000 MB
✅ dashboard_metric_availability.parquet        0.004 MB
✅ dashboard_metric_availability.csv            0.001 MB

✅ Сводная статистика построена только на готовых mart.
✅ mart_events.parquet не читался.
